# Lezione 6 — NLP I: tokenizzazione e sequenziazione del testo

Corso **"Fondamenti di TensorFlow da zero"** — DIH DATAlife, a partire dalla certificazione EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals).

## Obiettivi di questa lezione

- Capire perché le parole devono essere convertite in numeri per le reti neurali
- Implementare la tokenizzazione: dividere il testo in parole e assegnare a ogni parola un ID numerico univoco
- Costruire un vocabolario (vocabulary) e padroneggiare il problema dei token sconosciuti (out-of-vocabulary, OOV)
- Preparare sequenze di testo per modelli di NLP

## Parte 1 — Perché il testo ha bisogno di essere convertito in numeri

Una rete neurale sa solo moltiplicare, sommare e regolare numeri: non capisce le parole. Qualsiasi modello di elaborazione del linguaggio naturale (NLP, Natural Language Processing) inizia convertendo il testo in numeri.

## Parte 2 — Tokenizzare e costruire un vocabolario

**Tokenizzare** è dividere un testo in unità (token, di solito parole) e assegnare a ogni token un numero univoco. Se una parola appare più di una volta in un testo, riceve lo stesso ID ogni volta.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = [
    'I love my dog',
    'I love my cat',
    'You love my dog!'
]

# num_words limita el vocabulario a las 100 palabras más frecuentes
tokenizer = Tokenizer(num_words=100)

# fit_on_texts recorre el corpus y construye el vocabulario
tokenizer.fit_on_texts(sentences)

# word_index es el diccionario palabra -> token
word_index = tokenizer.word_index
print(word_index)

Nota due dettagli del vocabolario che hai appena stampato:

- Tutto è in minuscolo: il tokenizer ha convertito "Hello" in "hello". È uno scelta di design: evita che la stessa parola con maiuscole diverse abbia ID diversi
- Il token 0 è "<PAD>", il token 1 è "<OOV>" (Out-Of-Vocabulary, fuori dal vocabolario). Questi token riservati hanno ID bassi e significati speciali

## Parte 3 — Parole che il tokenizzatore non conosce (OOV)

Il vocabolario contiene solo le parole più frequenti (le prime 1000 del dataset di allenamento). Una parola che non appare nei dati di allenamento—ad es. "manatee" in una frase nuova—finisce nel token OOV.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

sentences = [
    'I love my dog',
    'I love my cat',
    'You love my dog!',
    'Do you think my dog is amazing?'
]

# oov_token reserva un token para las palabras que no estén en el vocabulario
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

# texts_to_sequences convierte cada frase en su secuencia de tokens
sequences = tokenizer.texts_to_sequences(sentences)

print("Word index:", word_index)
print("Secuencias:", sequences)

Osserva che `"<OOV>"` occupa il token numero 1: è la parola più importante del vocabolario di NSL. Ogni volta che il tokenizzatore incontra una parola che non conosce, la sostituisce con 1.

## Parte 4 — Pareggiare le lunghezze con padding

Le quattro frasi precedenti hanno lunghezze diverse (dopo la tokenizzazione):  
- "I love this movie" → 4 token  
- "This movie was great!" → 4 token  
- "What a waste" → 3 token  
- "Such an amazing film" → 4 token  

Le reti neurali si aspettano input di dimensione fissa. Padding aggiunge token `<PAD>` (ID 0) all'inizio o alla fine della sequenza per equalizzare le lunghezze.

In [ ]:
# maxlen fija la longitud final de todas las secuencias
# por defecto, el relleno (padding) se añade al principio de la secuencia ('pre')
padded = pad_sequences(sequences, maxlen=5)

print("Secuencias originales:", sequences)
print("Secuencias con padding:")
print(padded)

Andiamo a verificare che il token OOV funziona con frasi completamente nuove, che contengono parole che non abbiamo visto in allenamento.

In [ ]:
# Frases nuevas, con palabras que no estaban en el corpus de entrenamiento
test_data = [
    'i really love my dog',
    'my dog loves my manatee'
]

test_seq = tokenizer.texts_to_sequences(test_data)
print("Secuencias de prueba:", test_seq)

padded_test = pad_sequences(test_seq, maxlen=10)
print("Con padding:")
print(padded_test)

Parole come "really", "loves" o "manatee" non erano nel vocabolario, così che vengono sostituite da 1 (token OOV). Il tokenizzatore e il preprocessor funzionano così: in fase di produzione, anche le parole sconosciute verranno mappate a 1.

## Riepilogo della lezione

- Una rete neurale elabora solo numeri: il NLP inizia convertendo il testo in numeri tramite tokenizzazione
- Un vocabolario mappa parole a ID univoci; le parole sconosciute diventano il token OOV
- Padding equalizza le sequenze di diverse lunghezze aggiungendo token riservati
- Tokenizzazione, vocabolario e padding sono gli step preliminari per qualsiasi modello di NLP